### Connecting to Data

In [0]:
import pandas as pd

In [0]:
dfofficecommute = spark.table("workspace.default.officecommute").toPandas()
dfroute = spark.table("workspace.default.route").toPandas()
dfroutesegments = spark.table("workspace.default.routesegments").toPandas()

In [0]:
dfofficecommute.head()

,CommuteID,RouteID,Route SegmentID,Traffic Intensity,Start Time,End Time,Date
0,1,2,6,1,648,650,2025-09-18
1,2,2,7,3,650,654,2025-09-18
2,3,2,8,1,654,655,2025-09-18
3,4,2,9,2,655,658,2025-09-18
4,5,2,10,1,658,700,2025-09-18


### Analysis

In [0]:
%sql
select * from officecommute limit 10

CommuteID,RouteID,Route SegmentID,Traffic Intensity,Start Time,End Time,Date
1,2,6,1,648,650,2025-09-18T00:00:00.000Z
2,2,7,3,650,654,2025-09-18T00:00:00.000Z
3,2,8,1,654,655,2025-09-18T00:00:00.000Z
4,2,9,2,655,658,2025-09-18T00:00:00.000Z
5,2,10,1,658,700,2025-09-18T00:00:00.000Z
6,2,11,2,700,703,2025-09-18T00:00:00.000Z
7,1,1,2,717,720,2025-09-19T00:00:00.000Z
8,1,2,5,720,728,2025-09-19T00:00:00.000Z
9,1,3,5,728,737,2025-09-19T00:00:00.000Z
10,1,4,2,737,740,2025-09-19T00:00:00.000Z


In [0]:
%sql
--Average Commute Time in Minutes
WITH segment_times AS (
    SELECT
        CommuteID,
        RouteID,
        `Route SegmentID`,
        `Date`,
        `Start Time`,
        `End Time`,
        (CAST(`Start Time` / 100 AS INT) * 60) + (`Start Time` % 100) AS StartMinutes,
        (CAST(`End Time` / 100 AS INT) * 60) + (`End Time` % 100) AS EndMinutes
    FROM officecommute
),

segment_durations AS (
    SELECT
        CommuteID,
        RouteID,
        `Route SegmentID`,
        `Date`,
        `Start Time`,
        `End Time`,
        StartMinutes,
        EndMinutes,
        EndMinutes - StartMinutes AS SegmentDurationMinutes
    FROM segment_times
),

daily_commute_time AS (
    SELECT
        RouteID,
        `Date`,
        SUM(SegmentDurationMinutes) AS DailyCommuteMinutes
    FROM segment_durations
    GROUP BY
        RouteID,
        `Date`
)

SELECT
    RouteID,
    `Date`,
    DailyCommuteMinutes
FROM daily_commute_time
ORDER BY
    `Date`,
    RouteID;

RouteID,Date,DailyCommuteMinutes
2,2025-09-18T00:00:00.000Z,15
1,2025-09-19T00:00:00.000Z,27
2,2025-09-22T00:00:00.000Z,23
1,2025-09-23T00:00:00.000Z,11
2,2025-09-24T00:00:00.000Z,23
1,2025-09-25T00:00:00.000Z,9
2,2025-09-26T00:00:00.000Z,22
1,2025-09-29T00:00:00.000Z,25
2,2025-09-30T00:00:00.000Z,16
1,2025-10-01T00:00:00.000Z,16


In [0]:
%sql
-- Average Commute Time by Route
WITH daily_commute AS (
    SELECT
        RouteID,
        `Date`,
        SUM(((CAST(`End Time` / 100 AS INT) * 60) + (`End Time` % 100))
            -
            ((CAST(`Start Time` / 100 AS INT) * 60) + (`Start Time` % 100))) AS DailyCommuteMinutes
    FROM officecommute
    GROUP BY
        RouteID,
        `Date`
)
SELECT
    r.`Route Description`,
    CONCAT(
        CAST(FLOOR(AVG(dc.DailyCommuteMinutes)) AS INT), ' min ',
        CAST(ROUND((AVG(dc.DailyCommuteMinutes) - FLOOR(AVG(dc.DailyCommuteMinutes))) * 60, 0
            )AS INT), ' sec') AS AverageCommuteDuration
FROM daily_commute dc
JOIN route r ON dc.RouteID = r.RouteID

GROUP BY
    r.`Route Description`
ORDER BY
    r.`Route Description`

Route Description,AverageCommuteDuration
MST to PTA via BKR,16 min 18 sec
MST to PTA via MRH,17 min 57 sec


In [0]:
%sql
WITH daily_commute AS (
    SELECT
        RouteID,
        TO_DATE(`Date`, 'dd/MM/yyyy') AS CommuteDate,
        DATE_FORMAT(TO_DATE(`Date`, 'dd/MM/yyyy'), 'EEEE') AS DayName,
        DAYOFWEEK(TO_DATE(`Date`, 'dd/MM/yyyy')) AS DayNumber,
        SUM(
            ((CAST(`End Time` / 100 AS INT) * 60) + (`End Time` % 100))
            -
            ((CAST(`Start Time` / 100 AS INT) * 60) + (`Start Time` % 100))
        ) AS DailyCommuteMinutes
    FROM officecommute
    GROUP BY
        RouteID,
        TO_DATE(`Date`, 'dd/MM/yyyy')
),
average_by_day_route AS (
    SELECT
        DayName,
        DayNumber,
        AVG(CASE WHEN RouteID = 1 THEN DailyCommuteMinutes END) AS Route1Avg,
        AVG(CASE WHEN RouteID = 2 THEN DailyCommuteMinutes END) AS Route2Avg
    FROM daily_commute
    GROUP BY
        DayName,
        DayNumber
)
SELECT
    DayName AS `Day of Week`,
    CONCAT(
        CAST(FLOOR(Route1Avg) AS INT),' min ',
        CAST(ROUND((Route1Avg - FLOOR(Route1Avg)) * 60, 0) AS INT),' sec'
    ) AS `MST to PTA via BKR`,
    CONCAT(
        CAST(FLOOR(Route2Avg) AS INT),' min ',
        CAST(ROUND((Route2Avg - FLOOR(Route2Avg)) * 60, 0) AS INT),' sec'
    ) AS `MST to PTA via MRH`
FROM average_by_day_route
ORDER BY
    DayNumber;

Day of Week,MST to PTA via BKR,MST to PTA via MRH
Monday,16 min 3 sec,17 min 18 sec
Tuesday,16 min 27 sec,19 min 35 sec
Wednesday,14 min 30 sec,16 min 54 sec
Thursday,16 min 27 sec,18 min 51 sec
Friday,17 min 57 sec,17 min 9 sec


In [0]:
%sql
WITH daily_commute AS (
    SELECT
        RouteID,
        `Date`,
        MIN(`Start Time`) AS TripStartTime,
        SUM(
            ((CAST(`End Time` / 100 AS INT) * 60) + (`End Time` % 100))
            -
            ((CAST(`Start Time` / 100 AS INT) * 60) + (`Start Time` % 100))
        ) AS DailyCommuteMinutes,
        AVG(`Traffic Intensity`) AS AverageTrafficIntensity
    FROM officecommute
    GROUP BY
        RouteID,
        `Date`
),

bucketed_commute AS (
    SELECT
        RouteID,
        CASE
            WHEN TripStartTime BETWEEN 645 AND 700 THEN '06:45 to 07:00'
            WHEN TripStartTime BETWEEN 701 AND 715 THEN '07:01 to 07:15'
            WHEN TripStartTime BETWEEN 716 AND 730 THEN '07:16 to 07:30'
            WHEN TripStartTime BETWEEN 731 AND 745 THEN '07:31 to 07:45'
            ELSE 'Outside bucket'
        END AS StartTimeBucket,
        DailyCommuteMinutes,
        AverageTrafficIntensity
    FROM daily_commute
),

route_bucket_average AS (
    SELECT
        StartTimeBucket,
        RouteID,
        AVG(DailyCommuteMinutes) AS AverageCommuteMinutes,
        ROUND(AVG(AverageTrafficIntensity), 1) AS AverageTrafficIntensity
    FROM bucketed_commute
    WHERE StartTimeBucket <> 'Outside bucket'
    GROUP BY
        StartTimeBucket,
        RouteID
)

SELECT
    StartTimeBucket,
    CONCAT(
        CAST(FLOOR(MAX(CASE WHEN RouteID = 1 THEN AverageCommuteMinutes END)) AS INT),' min ',
        CAST(
            ROUND(
                (
                    MAX(CASE WHEN RouteID = 1 THEN AverageCommuteMinutes END)
                    - FLOOR(MAX(CASE WHEN RouteID = 1 THEN AverageCommuteMinutes END))
                ) * 60, 0) AS INT ),' sec'
    ) AS `MST to PTA via BKR Avg Commute Time`,

    CONCAT(
        CAST(FLOOR(MAX(CASE WHEN RouteID = 2 THEN AverageCommuteMinutes END)) AS INT),' min ',
        CAST(
            ROUND(
                (
                    MAX(CASE WHEN RouteID = 2 THEN AverageCommuteMinutes END)
                    - FLOOR(MAX(CASE WHEN RouteID = 2 THEN AverageCommuteMinutes END))
                ) * 60, 0) AS INT),' sec'
    ) AS `MST to PTA via MRH Avg Commute Time`
FROM route_bucket_average
GROUP BY
    StartTimeBucket
ORDER BY
    StartTimeBucket;

StartTimeBucket,MST to PTA via BKR Avg Commute Time,MST to PTA via MRH Avg Commute Time
06:45 to 07:00,11 min 49 sec,13 min 31 sec
07:01 to 07:15,19 min 10 sec,22 min 21 sec
07:16 to 07:30,19 min 42 sec,23 min 28 sec


In [0]:
%sql
WITH daily_commute AS (
    SELECT
        RouteID,
        TO_DATE(`Date`, 'dd/MM/yyyy') AS CommuteDate,
        DATE_FORMAT(TO_DATE(`Date`, 'dd/MM/yyyy'), 'EEEE') AS DayName,
        DAYOFWEEK(TO_DATE(`Date`, 'dd/MM/yyyy')) AS DayNumber,
        MIN(`Start Time`) AS TripStartTime,
        SUM(
            ((CAST(`End Time`/100 AS INT) * 60) + (`End Time` % 100))
            -
            ((CAST(`Start Time`/100 AS INT) * 60) + (`Start Time` % 100))
        ) AS DailyCommuteMinutes,
        AVG(`Traffic Intensity`) AS AverageTrafficIntensity
    FROM officecommute
    GROUP BY
        RouteID,
        TO_DATE(`Date`, 'dd/MM/yyyy')
),

bucketed_commute AS (
    SELECT
        RouteID,
        DayName,
        DayNumber,
        CASE
            WHEN TripStartTime BETWEEN 645 AND 700 THEN '06:45 to 07:00'
            WHEN TripStartTime BETWEEN 701 AND 715 THEN '07:01 to 07:15'
            WHEN TripStartTime BETWEEN 716 AND 730 THEN '07:16 to 07:30'
            WHEN TripStartTime BETWEEN 731 AND 745 THEN '07:31 to 07:45'
            ELSE 'Outside bucket'
        END AS StartTimeBucket,
        DailyCommuteMinutes,
        AverageTrafficIntensity
    FROM daily_commute
),

route_bucket_average AS (
    SELECT
        StartTimeBucket,
        DayName,
        DayNumber,
        RouteID,
        AVG(DailyCommuteMinutes) AS AverageCommuteMinutes,
        ROUND(AVG(AverageTrafficIntensity), 1) AS AverageTrafficIntensity
    FROM bucketed_commute
    WHERE StartTimeBucket <> 'Outside bucket'
    GROUP BY
        StartTimeBucket,
        DayName,
        DayNumber,
        RouteID
)

SELECT
    StartTimeBucket,
    DayName AS `Day of Week`,
    CONCAT(
        CAST(FLOOR(MAX(CASE WHEN RouteID = 1 THEN AverageCommuteMinutes END)) AS INT),
        ' min ',
        CAST(
            ROUND(
                (
                    MAX(CASE WHEN RouteID = 1 THEN AverageCommuteMinutes END)
                    - FLOOR(MAX(CASE WHEN RouteID = 1 THEN AverageCommuteMinutes END))
                ) * 60,
                0
            ) AS INT
        ),
        ' sec'
    ) AS `MST to PTA via BKR Avg Commute Time`,

    CONCAT(
        CAST(FLOOR(MAX(CASE WHEN RouteID = 2 THEN AverageCommuteMinutes END)) AS INT),
        ' min ',
        CAST(
            ROUND(
                (
                    MAX(CASE WHEN RouteID = 2 THEN AverageCommuteMinutes END)
                    - FLOOR(MAX(CASE WHEN RouteID = 2 THEN AverageCommuteMinutes END))
                ) * 60,
                0
            ) AS INT
        ),
        ' sec'
    ) AS `MST to PTA via MRH Avg Commute Time`

FROM route_bucket_average
GROUP BY
    StartTimeBucket,
    DayName,
    DayNumber
ORDER BY
    StartTimeBucket,
    DayNumber;

StartTimeBucket,Day of Week,MST to PTA via BKR Avg Commute Time,MST to PTA via MRH Avg Commute Time
06:45 to 07:00,Monday,12 min 23 sec,11 min 33 sec
06:45 to 07:00,Tuesday,12 min 30 sec,14 min 12 sec
06:45 to 07:00,Wednesday,10 min 49 sec,14 min 5 sec
06:45 to 07:00,Thursday,11 min 0 sec,12 min 53 sec
06:45 to 07:00,Friday,12 min 43 sec,14 min 20 sec
07:01 to 07:15,Monday,17 min 23 sec,22 min 15 sec
07:01 to 07:15,Tuesday,19 min 40 sec,25 min 45 sec
07:01 to 07:15,Wednesday,19 min 34 sec,22 min 10 sec
07:01 to 07:15,Thursday,19 min 22 sec,20 min 12 sec
07:01 to 07:15,Friday,19 min 35 sec,18 min 45 sec


In [0]:
%sql
-- Average Traffic Intensity by Route and Day of Week
SELECT
    r.`Route Description` AS Route,
    DATE_FORMAT(TO_DATE(oc.`Date`, 'dd/MM/yyyy'), 'EEEE') AS `Day of Week`,
    ROUND(AVG(oc.`Traffic Intensity`), 2) AS `Average Traffic Intensity`
FROM officecommute oc
JOIN route r
    ON oc.RouteID = r.RouteID
GROUP BY
    r.`Route Description`,
    DATE_FORMAT(TO_DATE(oc.`Date`, 'dd/MM/yyyy'), 'EEEE'),
    DAYOFWEEK(TO_DATE(oc.`Date`, 'dd/MM/yyyy'))
ORDER BY
    DAYOFWEEK(TO_DATE(oc.`Date`, 'dd/MM/yyyy')),
    Route;

Route,Day of Week,Average Traffic Intensity
MST to PTA via BKR,Monday,2.54
MST to PTA via MRH,Monday,2.12
MST to PTA via BKR,Tuesday,2.63
MST to PTA via MRH,Tuesday,2.34
MST to PTA via BKR,Wednesday,2.41
MST to PTA via MRH,Wednesday,2.09
MST to PTA via BKR,Thursday,2.5
MST to PTA via MRH,Thursday,2.16
MST to PTA via BKR,Friday,2.7
MST to PTA via MRH,Friday,2.18
